# Anveshan DataSprint 2.0 — Official v3 (`YOLO26m`) Inference, Benchmark & Submission Notebook

**Team:** `PavanKumar9938` (Team ID: `17424795`) | **Kaggle Public LB:** **`44.99750` (`#1` Overall, Sub `#58788362`)** | **1× T4 Speed:** `21.30 ms` val / `21.93 ms` test (`Speed_T4 = 1.0000`)

This command-driven Kaggle Notebook attaches our public **Kaggle Input Dataset** (`pavankumar9938/datasprint2-v3-yolo26m-weights`, containing `inference.py`, `weights/best.pt`, `requirements.txt`, `benchmark_speed.py`, `score.py`, `make_kaggle_submission.py`, and `check_submission.py`) alongside the official competition **`test_images/`** (and **`val_images/`** when mounted), verifies weights integrity (`SHA-256`), executes **`benchmark_speed.py` + `score.py`**, and outputs validated **`/kaggle/working/submission.csv`**.

---

### Official Links & Verification Artifacts

- **Public GitHub Repository (Full Source, Notebooks, Results & Git LFS Weights):** https://github.com/9553943884pavan-alt/datasprint-2-road-damage-detection
  - Pinned Commit SHA: [`1850a9cbee26e5b04298dd805e274c0442573d73`](https://github.com/9553943884pavan-alt/datasprint-2-road-damage-detection/commit/1850a9cbee26e5b04298dd805e274c0442573d73)
- **Public Kaggle Input Dataset (`inference.py` + `weights/best.pt` + official verifier scripts):** https://www.kaggle.com/datasets/pavankumar9938/datasprint2-v3-yolo26m-weights
- **2-Page Technical Write-Up PDF (Public View):** https://drive.google.com/file/d/1IlV-Sk9pc0iUJm908KhvY-yFaxclyMir/view?usp=sharing
- **Full Living Technical Report (Public View):** https://docs.google.com/document/d/1Zuy7Fj5oitn2LOqGKfjFYh6lm-pICSMqKajjc3_DTHU/view?usp=sharing
- **Experiment Tracker Sheet (Public View):** https://docs.google.com/spreadsheets/d/1vV_PznSUwoTXO1xN0gbigATpID_XKsJ7TdEHRu2XYP0/view?usp=sharing

### Compliance Declaration (Rules Section B.8.d)
- **Generic Pretrained Weights Only:** Initialized strictly from Ultralytics generic COCO checkpoint (`yolo26m.pt`); zero external road/crack/pavement weights and zero Objects365/DINOv3 weights used.
- **Competition Data Only:** Trained solely on official DataSprint 2.0 RDD2020 splits (`14,691` train / `2,150` val / `4,200` test); zero external datasets.


In [ ]:
import importlib.util
import platform
import subprocess
import sys

print("Python:", platform.python_version(), "| Platform:", platform.platform())

if importlib.util.find_spec("ultralytics") is None:
    print("Installing ultralytics==8.4.216 ...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
        "--upgrade-strategy", "only-if-needed", "ultralytics==8.4.216"
    ])
else:
    import ultralytics
    if ultralytics.__version__ != "8.4.216":
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
            "--upgrade-strategy", "only-if-needed", "ultralytics==8.4.216"
        ])

import torch
import ultralytics
print(
    f"torch={torch.__version__} | cuda_available={torch.cuda.is_available()} | "
    f"gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'} | "
    f"ultralytics={ultralytics.__version__}"
)


In [ ]:
import hashlib
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/9553943884pavan-alt/datasprint-2-road-damage-detection"
KAGGLE_DATASET_URL = "https://www.kaggle.com/datasets/pavankumar9938/datasprint2-v3-yolo26m-weights"
EXPECTED_WEIGHTS_SHA256 = "9f9fd2b5d51818759337daff442acdb8101ec2018be89b1deb31331d3cfde9a4"

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")

print("Public GitHub Repo:", GITHUB_REPO_URL)
print("Public Kaggle Weights/Scripts Dataset:", KAGGLE_DATASET_URL)
print("Scanning /kaggle/input:", [p.name for p in sorted(INPUT_ROOT.iterdir())] if INPUT_ROOT.exists() else [])

# 1. Discover attached Input Dataset (inference.py + weights/best.pt + benchmark/score/submission scripts)
REPO_DIR = None
for cand in sorted(INPUT_ROOT.rglob("inference.py")):
    parent = cand.parent
    if (parent / "weights" / "best.pt").is_file() and (parent / "benchmark_speed.py").is_file():
        REPO_DIR = parent
        break
assert REPO_DIR is not None, f"Could not locate inference.py + weights/best.pt + benchmark_speed.py under {INPUT_ROOT}"

WEIGHTS_PATH = REPO_DIR / "weights" / "best.pt"
actual_sha256 = hashlib.sha256(WEIGHTS_PATH.read_bytes()).hexdigest()
print(f"Verified REPO_DIR: {REPO_DIR}")
print(f"Verified weights/best.pt: {WEIGHTS_PATH.stat().st_size:,} B | sha256={actual_sha256}")
assert actual_sha256 == EXPECTED_WEIGHTS_SHA256, f"Weights SHA-256 mismatch: {actual_sha256}"

# 2. Discover official competition data directory (test_images + test_image_ids.csv)
COMP_DIR = None
for cand in sorted(INPUT_ROOT.rglob("test_image_ids.csv")):
    parent = cand.parent
    if (parent / "test_images").is_dir():
        COMP_DIR = parent
        break
assert COMP_DIR is not None, f"Could not locate competition data dir with test_images under {INPUT_ROOT}"

TEST_IMAGES_DIR = COMP_DIR / "test_images"
TEST_IDS_CSV = COMP_DIR / "test_image_ids.csv"
VAL_IMAGES_DIR = COMP_DIR / "val_images" if (COMP_DIR / "val_images").is_dir() else None
VAL_LABELS_CSV = COMP_DIR / "val_labels.csv" if (COMP_DIR / "val_labels.csv").is_file() else None
VAL_IDS_CSV = COMP_DIR / "val_image_ids.csv" if (COMP_DIR / "val_image_ids.csv").is_file() else None

BENCHMARK_PY = REPO_DIR / "benchmark_speed.py"
SCORE_PY = REPO_DIR / "score.py"
MAKE_SUB_PY = REPO_DIR / "make_kaggle_submission.py"
CHECK_SUB_PY = REPO_DIR / "check_submission.py"

print(f"Competition dir: {COMP_DIR} | test_images={len(list(TEST_IMAGES_DIR.glob('*.jpg')))}")
if VAL_IMAGES_DIR is not None:
    print(f"val_images mounted: {len(list(VAL_IMAGES_DIR.glob('*.jpg')))}")
else:
    print("val_images not present in competition input mount; skipping val step and running test benchmark.")


In [ ]:
import json
import subprocess
import sys

VAL_PRED_CSV = WORK_ROOT / "val_pred.csv"
VAL_SPEED_JSON = WORK_ROOT / "val_speed.json"

if VAL_IMAGES_DIR is not None and VAL_LABELS_CSV is not None and VAL_IDS_CSV is not None:
    cmd_bench = [
        sys.executable, str(BENCHMARK_PY),
        "--repo", str(REPO_DIR),
        "--images", str(VAL_IMAGES_DIR),
        "--out-csv", str(VAL_PRED_CSV),
        "--out-json", str(VAL_SPEED_JSON),
        "--warmup", "20",
        "--runs", "2",
    ]
    print("EXEC:", " ".join(cmd_bench))
    subprocess.run(cmd_bench, check=True)

    cmd_score = [
        sys.executable, str(SCORE_PY),
        "--pred", str(VAL_PRED_CSV),
        "--gt", str(VAL_LABELS_CSV),
        "--image-ids", str(VAL_IDS_CSV),
    ]
    print("\nEXEC:", " ".join(cmd_score))
    subprocess.run(cmd_score, check=True)

    print("\nVAL SPEED JSON:", json.dumps(json.loads(VAL_SPEED_JSON.read_text(encoding="utf-8")), indent=2))
else:
    print("Validation split not mounted; proceeding directly to test_images benchmark & submission.")


In [ ]:
import json
import pandas as pd
import subprocess
import sys

TEST_PRED_CSV = WORK_ROOT / "test_pred.csv"
TEST_SPEED_JSON = WORK_ROOT / "test_speed.json"
SUBMISSION_CSV = WORK_ROOT / "submission.csv"

# 1. Execute benchmark_speed.py directly against REPO_DIR (imports inference.py + loads weights/best.pt)
cmd_test_bench = [
    sys.executable, str(BENCHMARK_PY),
    "--repo", str(REPO_DIR),
    "--images", str(TEST_IMAGES_DIR),
    "--out-csv", str(TEST_PRED_CSV),
    "--out-json", str(TEST_SPEED_JSON),
    "--warmup", "10",
    "--runs", "1",
]
print("EXEC:", " ".join(cmd_test_bench))
subprocess.run(cmd_test_bench, check=True)

test_speed = json.loads(TEST_SPEED_JSON.read_text(encoding="utf-8"))
print("\nTEST SPEED JSON:", json.dumps(test_speed, indent=2))

# 2. Execute make_kaggle_submission.py
cmd_make_sub = [
    sys.executable, str(MAKE_SUB_PY),
    "--pred", str(TEST_PRED_CSV),
    "--image-ids", str(TEST_IDS_CSV),
    "--out", str(SUBMISSION_CSV),
]
print("\nEXEC:", " ".join(cmd_make_sub))
subprocess.run(cmd_make_sub, check=True)

# 3. Execute check_submission.py
cmd_check_sub = [
    sys.executable, str(CHECK_SUB_PY),
    "--pred", str(SUBMISSION_CSV),
    "--image-ids", str(TEST_IDS_CSV),
]
print("\nEXEC:", " ".join(cmd_check_sub))
subprocess.run(cmd_check_sub, check=True)

sub_df = pd.read_csv(SUBMISSION_CSV)
n_none = int((sub_df["PredictionString"] == "none").sum())
print(
    f"\nFINAL VERIFIED SUBMISSION: {SUBMISSION_CSV} | rows={len(sub_df)} | "
    f"size={SUBMISSION_CSV.stat().st_size:,} B | empty('none')={n_none} | "
    f"non-empty={len(sub_df) - n_none}"
)
print("Public GitHub Repo:", GITHUB_REPO_URL)
print("Public Kaggle Input Dataset:", KAGGLE_DATASET_URL)
print(sub_df.head(5))
